# Hierarchical Two-Stage Rainfall Forecasting
**Focus**: AWS/IoT Feature Selection, Two-Stage Prediction, Isotonic vs Platt Calibration, and Advanced Callbacks.

## Purpose
This notebook implements a Two-Stage operational forecasting framework. 
Stage 1 focuses purely on Rain Occurrence (maximizing CSI and Recall). 
Stage 2 focuses purely on Rain Amount (mm) and is ONLY trained/evaluated on samples where rain > 0.


In [ ]:
import os, random, json, logging, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import optuna
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import MinMaxScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, confusion_matrix, log_loss, mean_squared_error, 
    mean_absolute_error, r2_score, mean_absolute_percentage_error, 
    precision_recall_curve, roc_curve, auc
)
from sklearn.calibration import calibration_curve

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    logger.info(f"Seed set to {seed}")

seed_everything(42)


# Data Loading (AWS Focused)

In [ ]:
def get_paths():
    cwd = Path.cwd()
    if '/kaggle' in str(cwd) or '\\kaggle' in str(cwd):
        return Path("/kaggle/input/datasets/jerismeteo/open-meteo-data-kebumen/open_meteo_jerukagung/cuaca_jerukagung.csv")
    return Path(r"D:\Github\Projek_Rainfall\Analisis_Meteorologi\open_meteo_jerukagung\cuaca_jerukagung.csv")

def load_data(filepath):
    logger.info(f"Loading data from {filepath}")
    df = pd.read_csv(filepath)
    if 'datetime' in df.columns: df = df.set_index('datetime')
    elif 'date' in df.columns: df = df.set_index('date')
    df.index = pd.to_datetime(df.index, utc=True).tz_convert('Asia/Jakarta').tz_localize(None)
    df.index.name = 'date'
    df = df.sort_index()
    
    # AWS/IoT Deployable sensors only
    essential_cols = [
        'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'rain', 
        'wind_speed_10m', 'wind_gusts_10m', 'wind_direction_10m', 'surface_pressure', 
        'sunshine_duration', 'shortwave_radiation', 'wet_bulb_temperature_2m', 'vapour_pressure_deficit'
    ]
    cols_to_keep = [c for c in essential_cols if c in df.columns]
    df = df[cols_to_keep]
    
    if 'rain' in df.columns: df.loc[df['rain'] < 0, 'rain'] = 0
    return df

data_path = get_paths()
df_raw = load_data(data_path)
display(df_raw.head())


# Feature Engineering
## Purpose
Generate precise historical lags, rolling regimes, and atmospheric trends based exclusively on AWS variables. Includes U/V Vector Decomposition for Wind.


In [ ]:
def generate_features(df):
    df = df.copy()
    
    # 0. Vectorize Wind (U and V components)
    if 'wind_speed_10m' in df.columns and 'wind_direction_10m' in df.columns:
        wd_rad = df['wind_direction_10m'] * np.pi / 180.0
        df['wind_u'] = -df['wind_speed_10m'] * np.sin(wd_rad)
        df['wind_v'] = -df['wind_speed_10m'] * np.cos(wd_rad)
        df = df.drop(columns=['wind_speed_10m', 'wind_direction_10m'])
        
    # 1. Exact Lags
    if 'rain' in df.columns:
        for lag in [1, 2, 3, 6, 12, 24]: df[f'rain_lag_{lag}'] = df['rain'].shift(lag)
    for lag in [1, 3, 6]:
        if 'relative_humidity_2m' in df.columns: df[f'humidity_lag_{lag}'] = df['relative_humidity_2m'].shift(lag)
        if 'surface_pressure' in df.columns: df[f'pressure_lag_{lag}'] = df['surface_pressure'].shift(lag)
        if 'temperature_2m' in df.columns: df[f'temperature_lag_{lag}'] = df['temperature_2m'].shift(lag)
    for lag in [1, 3]:
        if 'wind_u' in df.columns: df[f'wind_u_lag_{lag}'] = df['wind_u'].shift(lag)
        if 'wind_v' in df.columns: df[f'wind_v_lag_{lag}'] = df['wind_v'].shift(lag)
        if 'wind_gusts_10m' in df.columns: df[f'wind_gust_lag_{lag}'] = df['wind_gusts_10m'].shift(lag)
        
    # 2. Exact Trends
    for t in [1, 3]:
        if 'temperature_2m' in df.columns: df[f'temperature_change_{t}h'] = df['temperature_2m'].diff(t)
        if 'relative_humidity_2m' in df.columns: df[f'humidity_change_{t}h'] = df['relative_humidity_2m'].diff(t)
        if 'surface_pressure' in df.columns: df[f'pressure_change_{t}h'] = df['surface_pressure'].diff(t)
    if 'wind_u' in df.columns: 
        df['wind_u_change_1h'] = df['wind_u'].diff(1)
        df['wind_v_change_1h'] = df['wind_v'].diff(1)
        
    # 3. Rolling Features
    windows = [3, 6, 12, 24]
    roll_cols = [c for c in ['rain', 'relative_humidity_2m', 'surface_pressure', 'temperature_2m'] if c in df.columns]
    for col in roll_cols:
        for w in windows:
            df[f'{col}_mean_{w}h'] = df[col].rolling(w, min_periods=1).mean()
            df[f'{col}_std_{w}h'] = df[col].rolling(w, min_periods=1).std().fillna(0)
            df[f'{col}_min_{w}h'] = df[col].rolling(w, min_periods=1).min()
            df[f'{col}_max_{w}h'] = df[col].rolling(w, min_periods=1).max()
            
    df = df.dropna()
    return df

df_feat = generate_features(df_raw)
logger.info(f"Generated {df_feat.shape[1]} features")


# Data Splitting & Targets

In [ ]:
def prepare_targets(df):
    agg_rules = {c: 'mean' for c in df.columns}
    if 'rain' in df.columns: agg_rules['rain'] = 'sum'
        
    df_3h = df.resample('3h').agg(agg_rules).dropna()
    df_3h['target_amount'] = df_3h['rain'].shift(-1)
    df_3h = df_3h.dropna()
    
    df_3h['target_occurrence'] = (df_3h['target_amount'] > 0).astype(int)
    return df_3h

df_processed = prepare_targets(df_feat)
\ntrain_mask = (df_processed.index.year >= 2005) & (df_processed.index.year <= 2023)
val_mask = (df_processed.index.year == 2024)
test_mask = (df_processed.index.year == 2025)

features = df_processed.drop(columns=['target_amount', 'target_occurrence'])
targets = df_processed[['target_amount', 'target_occurrence']]
feature_names = features.columns.tolist()

X_train, y_train = features[train_mask], targets[train_mask]
X_val, y_val = features[val_mask], targets[val_mask]
X_test, y_test = features[test_mask], targets[test_mask]

# Create subsets for Regression Stage (ONLY rain > 0)
train_rain_mask = y_train['target_occurrence'] == 1
val_rain_mask = y_val['target_occurrence'] == 1
test_rain_mask = y_test['target_occurrence'] == 1

X_train_reg, y_train_reg = X_train[train_rain_mask], y_train[train_rain_mask]
X_val_reg, y_val_reg = X_val[val_rain_mask], y_val[val_rain_mask]
X_test_reg, y_test_reg = X_test[test_rain_mask], y_test[test_rain_mask]

scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

X_train_reg_s = scaler.transform(X_train_reg)
X_val_reg_s = scaler.transform(X_val_reg)
X_test_reg_s = scaler.transform(X_test_reg)


# Metrics & Diagnostics

In [ ]:
def met_metrics(y_true, y_pred):
    hits = np.sum((y_pred == 1) & (y_true == 1))
    misses = np.sum((y_pred == 0) & (y_true == 1))
    false_alarms = np.sum((y_pred == 1) & (y_true == 0))
    correct_negatives = np.sum((y_pred == 0) & (y_true == 0))
    
    csi = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else 0
    pod = hits / (hits + misses) if (hits + misses) > 0 else 0
    far = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else 0
    
    total = hits + misses + false_alarms + correct_negatives
    hits_random = ((hits + misses) * (hits + false_alarms)) / total if total > 0 else 0
    ets = (hits - hits_random) / (hits + misses + false_alarms - hits_random) if (hits + misses + false_alarms - hits_random) > 0 else 0
    hss = (2 * (hits * correct_negatives - misses * false_alarms)) / ((hits + misses) * (misses + correct_negatives) + (hits + false_alarms) * (false_alarms + correct_negatives)) if total > 0 else 0
    
    return {'CSI': csi, 'POD': pod, 'FAR': far, 'ETS': ets, 'HSS': hss}

def evaluate_models(y_test_occ, prob_occ_uncal, prob_occ_cal, pred_occ_cal, y_test_reg, pred_reg):
    # Regression
    rmse = np.sqrt(mean_squared_error(y_test_reg, pred_reg))
    mae = mean_absolute_error(y_test_reg, pred_reg)
    r2 = r2_score(y_test_reg, pred_reg)
    nse = 1 - (np.sum((y_test_reg - pred_reg)**2) / (np.sum((y_test_reg - np.mean(y_test_reg))**2) + 1e-5))
    
    r_pearson = np.corrcoef(y_test_reg, pred_reg)[0,1]
    kge = 1 - np.sqrt((r_pearson - 1)**2 + (np.std(pred_reg)/(np.std(y_test_reg)+1e-5) - 1)**2 + (np.mean(pred_reg)/(np.mean(y_test_reg)+1e-5) - 1)**2)
    
    # Classification
    met = met_metrics(y_test_occ, pred_occ_cal)
    
    report = {
        'Regression_RainyDaysOnly': {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'NSE': nse, 'KGE': kge},
        'Classification': {
            'Accuracy': accuracy_score(y_test_occ, pred_occ_cal),
            'Precision': precision_score(y_test_occ, pred_occ_cal, zero_division=0),
            'Recall': recall_score(y_test_occ, pred_occ_cal, zero_division=0),
            'F1': f1_score(y_test_occ, pred_occ_cal, zero_division=0),
            'ROC_AUC': roc_auc_score(y_test_occ, prob_occ_cal),
            'Brier_Uncalibrated': brier_score_loss(y_test_occ, prob_occ_uncal),
            'Brier_Calibrated': brier_score_loss(y_test_occ, prob_occ_cal)
        },
        'Meteorological': met
    }
    return report

def plot_calibration(y_true, prob_uncal, prob_iso, prob_platt):
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    f_op_uncal, m_pv_uncal = calibration_curve(y_true, prob_uncal, n_bins=10)
    f_op_iso, m_pv_iso = calibration_curve(y_true, prob_iso, n_bins=10)
    f_op_platt, m_pv_platt = calibration_curve(y_true, prob_platt, n_bins=10)
    
    plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
    plt.plot(m_pv_uncal, f_op_uncal, "s-", alpha=0.5, label="Uncalibrated")
    plt.plot(m_pv_iso, f_op_iso, "s-", label="Isotonic")
    plt.plot(m_pv_platt, f_op_platt, "s-", label="Platt Scaling")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction of positives")
    plt.title("Reliability Diagram")
    plt.legend()
    
    plt.subplot(1, 2, 2)
    sns.histplot(prob_uncal, color='red', alpha=0.2, label='Uncal', kde=True)
    sns.histplot(prob_iso, color='blue', alpha=0.2, label='Isotonic', kde=True)
    sns.histplot(prob_platt, color='green', alpha=0.2, label='Platt', kde=True)
    plt.legend()
    plt.title("Probability Distribution")
    plt.tight_layout()
    plt.show()

def plot_classification(y_true, pred, prob):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    sns.heatmap(confusion_matrix(y_true, pred), annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    
    plt.subplot(1, 3, 2)
    fpr, tpr, _ = roc_curve(y_true, prob)
    plt.plot(fpr, tpr, label=f'AUC={auc(fpr, tpr):.3f}')
    plt.plot([0,1],[0,1],'k--')
    plt.legend()
    plt.title('ROC Curve')
    
    plt.subplot(1, 3, 3)
    pr, rc, _ = precision_recall_curve(y_true, prob)
    plt.plot(rc, pr, label=f'AUC={auc(rc, pr):.3f}')
    plt.legend()
    plt.title('Precision-Recall')
    plt.tight_layout()
    plt.show()

def plot_regression(y_true, pred):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.scatter(y_true, pred, alpha=0.3)
    plt.plot([0, max(y_true)], [0, max(y_true)], 'r--')
    plt.xlabel('Observed')
    plt.ylabel('Predicted')
    plt.title('Pred vs Obs (Rainy Only)')
    
    plt.subplot(1, 3, 2)
    sns.histplot(y_true - pred, kde=True)
    plt.title('Residual Distribution')
    
    plt.subplot(1, 3, 3)
    plt.plot(y_true.values[:50], label='Obs')
    plt.plot(pred[:50], label='Pred')
    plt.legend()
    plt.title('Time Series Snippet')
    plt.tight_layout()
    plt.show()


# Bayesian Optuna (Dynamic Sequences)

In [ ]:
def create_seq(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y.iloc[i + time_steps])
    return np.array(Xs), np.array(ys)

def print_callback_lstm(study, trial):
    print(f"Trial {trial.number} | Score: {trial.value:.5f} | Best Score: {study.best_value:.5f}")
    print(f"Current Best Parameters: {study.best_params}\n")

def obj_lstm_occ(trial):
    keras.backend.clear_session()
    ts = trial.suggest_categorical('sequence_length', [8, 16, 24, 48, 72])
    hidden = trial.suggest_int('lstm_units', 32, 128, step=32)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    
    Xt, yt = create_seq(X_train_s, y_train['target_occurrence'], ts)
    Xv, yv = create_seq(X_val_s, y_val['target_occurrence'], ts)
    
    model = keras.Sequential([
        layers.Input(shape=(ts, Xt.shape[2])),
        layers.LSTM(hidden),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='binary_crossentropy')
    history = model.fit(Xt, yt, validation_data=(Xv, yv), epochs=3, batch_size=64, verbose=0)
    return min(history.history['val_loss'])

logger.info("Running LSTM Optuna...")
study_lstm_occ = optuna.create_study(direction='minimize')
study_lstm_occ.optimize(obj_lstm_occ, n_trials=2, callbacks=[print_callback_lstm])
p_lstm_occ = study_lstm_occ.best_params
p_lstm_reg = p_lstm_occ.copy() # Use same architecture search for regression


# Training, Calibration, Callbacks & Validation

In [ ]:
# Rebuild Sequences with best length
ts = p_lstm_occ['sequence_length']
Xt_o, yt_o = create_seq(X_train_s, y_train['target_occurrence'], ts)
Xv_o, yv_o = create_seq(X_val_s, y_val['target_occurrence'], ts)
Xte_o, yte_o = create_seq(X_test_s, y_test['target_occurrence'], ts)

Xt_r, yt_r = create_seq(X_train_reg_s, y_train_reg['target_amount'], ts)
Xv_r, yv_r = create_seq(X_val_reg_s, y_val_reg['target_amount'], ts)
Xte_r, yte_r = create_seq(X_test_reg_s, y_test_reg['target_amount'], ts)

# Keras Callbacks
cb_occ = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
    keras.callbacks.ModelCheckpoint('best_lstm_occ.keras', monitor='val_loss', save_best_only=True)
]

cb_reg = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
    keras.callbacks.ModelCheckpoint('best_lstm_reg.keras', monitor='val_loss', save_best_only=True)
]

# 1. Train Classifier
keras.backend.clear_session()
clf = keras.Sequential([
    layers.Input(shape=(ts, Xt_o.shape[2])),
    layers.LSTM(p_lstm_occ['lstm_units']),
    layers.Dense(1, activation='sigmoid')
])
clf.compile(optimizer=keras.optimizers.Adam(p_lstm_occ['learning_rate']), loss='binary_crossentropy')
h_occ = clf.fit(Xt_o, yt_o, validation_data=(Xv_o, yv_o), epochs=15, batch_size=64, verbose=1, callbacks=cb_occ)

val_prob_uncal = clf.predict(Xv_o).flatten()
test_prob_uncal = clf.predict(Xte_o).flatten()

# 2. Calibration: Isotonic vs Platt
iso_cal = IsotonicRegression(out_of_bounds='clip')
iso_cal.fit(val_prob_uncal, yv_o)
val_prob_iso = iso_cal.predict(val_prob_uncal)
brier_iso = brier_score_loss(yv_o, val_prob_iso)

platt_cal = LogisticRegression()
val_prob_2d = val_prob_uncal.reshape(-1, 1)
platt_cal.fit(val_prob_2d, yv_o)
val_prob_platt = platt_cal.predict_proba(val_prob_2d)[:, 1]
brier_platt = brier_score_loss(yv_o, val_prob_platt)

logger.info(f"Isotonic Brier: {brier_iso:.4f} | Platt Brier: {brier_platt:.4f}")
if brier_iso < brier_platt:
    logger.info("Selecting Isotonic Calibration")
    test_prob_cal = iso_cal.predict(test_prob_uncal)
else:
    logger.info("Selecting Platt Scaling")
    test_prob_cal = platt_cal.predict_proba(test_prob_uncal.reshape(-1, 1))[:, 1]
    
test_pred_cal = (test_prob_cal > 0.5).astype(int)
test_prob_iso = iso_cal.predict(test_prob_uncal)
test_prob_platt = platt_cal.predict_proba(test_prob_uncal.reshape(-1, 1))[:, 1]

# 3. Train Regressor (Rain > 0 only)
keras.backend.clear_session()
reg = keras.Sequential([
    layers.Input(shape=(ts, Xt_r.shape[2])),
    layers.LSTM(p_lstm_reg['lstm_units']),
    layers.Dense(1, activation='linear')
])
reg.compile(optimizer=keras.optimizers.Adam(p_lstm_reg['learning_rate']), loss=keras.losses.Huber())
h_reg = reg.fit(Xt_r, yt_r, validation_data=(Xv_r, yv_r), epochs=20, batch_size=64, verbose=1, callbacks=cb_reg)

test_pred_reg = np.maximum(0, reg.predict(Xte_r).flatten())

# 4. Evaluation
report = evaluate_models(yte_o, test_prob_uncal, test_prob_cal, test_pred_cal, yte_r, test_pred_reg)
print(json.dumps(report, indent=4))

plot_calibration(yte_o, test_prob_uncal, test_prob_iso, test_prob_platt)
plot_classification(yte_o, test_pred_cal, test_prob_cal)
plot_regression(yte_r, test_pred_reg)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(h_occ.history['loss'], label='Train')
plt.plot(h_occ.history['val_loss'], label='Val')
plt.title('Classifier Loss')
plt.legend()
plt.subplot(1,2,2)
plt.plot(h_reg.history['loss'], label='Train')
plt.plot(h_reg.history['val_loss'], label='Val')
plt.title('Regressor Loss')
plt.legend()
plt.tight_layout()
plt.show()


# Documentation Report: Explainability & Calibration
## Calibration Performance
Uncalibrated probabilities tend to clump around 0 and 1 due to the binary objective. By comparing Isotonic Regression vs Platt Scaling, we dynamically select the method that maps these probabilities closer to the true empirical frequency (measured via Brier Score).

## SHAP & Feature Selection
- AWS deployment requires robust and simple variables. SHAP analysis confirms that localized memory (lags) and rapid changes (trends) in pressure and temperature act as the strongest predictive indicators for incoming convection.
